# EuroSAT: when does NoLA help?
Run in **Robotics Research Lab**. This is a separate exploratory experiment for the GTX 1050 Ti, not a reproduction of the paper schedule. Use the training split for adaptation and the same validation subset for all comparisons. **This notebook never scores test images.**

Two questions: (1) does full NoLA beat unprompted CLIP CDE on validation? (2) does tuning CLIP improve on the fitted DINO head, or would we be better off stopping there? The table answers both per setting. Repeat with another seed before trusting a threshold, then lock a recipe and evaluate it on the untouched test split separately.

## 1. Set the sweep
Change `train_sizes` and `anchor_caps` here. The smaller adaptation set is a prefix of the larger one after a seeded shuffle. Validation rows stay fixed across settings and adaptation seeds. DINO uses a fixed number of full-batch head updates; CLIP uses the same number of minibatch updates in every setting. A smaller set is therefore revisited more times. To compare larger sets, increase `clip_steps` enough that `clip_steps × batch_size` covers the largest set at least once. Run the grid yourself in §4.

In [ ]:
from pathlib import Path
import itertools, json, random, sys
import pandas as pd

import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from IPython.display import display

root = Path.home() / "nola-walkthrough"
upstream = root / "upstream"
sys.dont_write_bytecode = True
sys.path[:0] = [str(upstream), str(root / ".deps")]
from clip import clip
from utils.data_utils import ds_specific_templates
from utils.model_utils import te_transform, tr_transforms, top_k_indices_per_class2


train_sizes = [160, 320]
anchor_caps = [4, 12]
seed = 42
val_n = 240
head_steps = 20
clip_steps = 80
batch_size = 4
n_prompts = 50
clip_lr = 0.001

image_dir = root / "data/eurosat/EuroSAT_RGB"
split = json.loads((image_dir.parent / "split_zhou_EuroSAT.json").read_text())
name_by_id = {label: name for _, label, name in split["train"]}
names = [name_by_id[i] for i in sorted(name_by_id)]
pool = random.Random(seed).sample(split["train"], max(train_sizes))
val = random.Random(123).sample(split["val"], val_n)
audit = torch.tensor([row[1] for row in pool])
y_val = torch.tensor([row[1] for row in val])


def load_images(rows, transform):
    batch = []
    for path, _, _ in rows:
        with Image.open(image_dir / path) as im:
            batch.append(transform(im.convert("RGB")))
    return torch.stack(batch).cuda()


print(f"{len(train_sizes) * len(anchor_caps)} runs; train sizes={train_sizes}; caps={anchor_caps}")
print(f"Each run: {head_steps} DINO-head updates, {clip_steps} CLIP updates, {val_n} validation images")

## 2. Score the pool and cache DINO features once
CLIP builds its CDE weights and scores the adaptation pool plus validation images. DINO encodes the same rows. Both models use the deterministic weak view here. Every grid row then reuses the same feature and score rows; there is no DINO encoder training. We move CLIP to CPU during DINO extraction and release DINO before moving CLIP back. The temporary `utils` swap avoids an import name collision between the local checkouts.

In [ ]:
model, width, preprocess = clip.load("ViT-B/32", device="cuda", download_root=str(root / "weights"))
model.float().eval()
descriptions = json.loads((upstream / "descriptions/generic/EuroSAT.json").read_text())
templates = ds_specific_templates["EuroSAT"]

weights = []
with torch.no_grad():
    for name in names:
        texts = descriptions[name] + [t.format(name) for t in templates]
        tokens = clip.tokenize(texts).cuda()
        vectors = F.normalize(model.encode_text(tokens), dim=1)
        weights.append(F.normalize(vectors.mean(dim=0), dim=0))
cde = torch.stack(weights, dim=1)
print("CDE weights [features, classes]:", tuple(cde.shape))

@torch.no_grad()
def clip_scores(rows):
    scores = []
    for i in range(0, len(rows), 8):
        features = F.normalize(model.encode_image(load_images(rows[i:i+8], te_transform)), dim=1)
        scores.append((model.logit_scale.exp() * (features @ cde)).softmax(dim=1).cpu())
    return torch.cat(scores)


pool_probs, val_probs = clip_scores(pool), clip_scores(val)
cde_correct = int((val_probs.argmax(1) == y_val).sum())
print(f"CDE baseline: {cde_correct}/{val_n}")

In [ ]:
model.cpu()
torch.cuda.empty_cache()
nola_utils = sys.modules.pop("utils")
try:
    dino = torch.hub.load(str(root / "dino-source"), "dino_vitb16", source="local", pretrained=False)
finally:
    sys.modules["utils"] = nola_utils
state = torch.load(root / "weights/dino_vitbase16_pretrain.pth", map_location="cpu", weights_only=True)
dino.load_state_dict(state)
del state
dino.eval().requires_grad_(False).cuda()


@torch.no_grad()
def dino_features(rows):
    return torch.cat([dino(load_images(rows[i:i+8], te_transform)).cpu()
                      for i in range(0, len(rows), 8)])



pool_features = dino_features(pool)
val_features = dino_features(val)
del dino
torch.cuda.empty_cache()
model.cuda()
print("Cached DINO features:", tuple(pool_features.shape), tuple(val_features.shape))

## 3. Define one matched run
Reuse the released top-k selector. It uses the adaptation labels to discover present classes and print audit statistics; the head and CLIP train on pseudo-labels, not those true labels. We count each returned label after fallback and deduplication. Each run starts from fresh prompts, fresh heads, and the original CLIP normalization weights. A fixed seed pairs the initializations across settings. The DINO head trains on selected anchors; prompted CLIP trains against DINO labels for the chosen adaptation subset.

**Augmentations.** `tr_transforms(image)` returns `[weak, strong]`. NoLA uses a deterministic resize and center crop for the DINO label, and the strong view for CLIP training. The strong view randomly requests a crop covering 20–100% of the image area and resizes it to 224×224, flips horizontally with probability 0.5, applies optional color jitter (0.8) and perspective warp (0.5), then always applies another random color jitter and Gaussian blur. Both views use CLIP channel normalization. Strong crops can sometimes remove class evidence; this sweep keeps augmentation fixed rather than varying it alongside training size and anchor cap.

In [ ]:
for name, param in model.named_parameters():
    param.requires_grad_("visual" in name and ("ln" in name or "bn" in name))
trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
initial_visual = [p.detach().clone() for _, p in trainable]
hidden = model.visual.class_embedding.numel()
bound = (6 / (3 * 16 * 16 + hidden)) ** 0.5


def strong(image):
    return tr_transforms(image)[1]


def run_setting(n, k):
    random.seed(seed)
    torch.manual_seed(seed)
    try:
        indices, _, pseudo = top_k_indices_per_class2(
            {"total_emb": pool_probs[:n], "idxs": torch.arange(n), "labels": audit[:n]}, k)
    except NotImplementedError:
        return dict(train_n=n, max_anchors=k, status="selector found no top-3 candidate for a class")

    dino_head = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, len(names)))
    opt = torch.optim.Adam(dino_head.parameters())
    for _ in range(head_steps):
        loss = F.cross_entropy(dino_head(pool_features[indices]), pseudo)
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        dino_labels = dino_head(pool_features[:n]).argmax(1)
        head_correct = int((dino_head(val_features).argmax(1) == y_val).sum())
        for (_, p), original in zip(trainable, initial_visual):
            p.copy_(original)

    prompt = nn.Parameter(torch.empty(1, n_prompts, hidden, device="cuda").uniform_(-bound, bound))
    classifier = nn.Linear(width, len(names), bias=False).cuda()
    with torch.no_grad():
        classifier.weight.copy_(cde.T)

    def prompted_logits(images):
        tokens = model.visual.embeddings_patch(images)
        tokens = torch.cat([tokens[:, :1], prompt.expand(len(images), -1, -1), tokens[:, 1:]], dim=1)
        return classifier(model.visual.forward_after_patch_embeddings(tokens))

    opt = torch.optim.AdamW([
        {"params": [prompt, *classifier.parameters(), *(p for name, p in trainable if "bias" not in name)]},
        {"params": [p for name, p in trainable if "bias" in name], "weight_decay": 0.0},
    ], lr=clip_lr)

    def batches():
        while True:
            yield from torch.randperm(n).split(batch_size)

    for ids in itertools.islice(batches(), clip_steps):
        images = load_images([pool[i] for i in ids.tolist()], strong)
        loss = F.cross_entropy(prompted_logits(images), dino_labels[ids].cuda(), label_smoothing=0.1)
        opt.zero_grad()
        loss.backward()
        opt.step()

    with torch.no_grad():
        pred = torch.cat([prompted_logits(load_images(val[i:i+8], te_transform)).argmax(1).cpu()
                          for i in range(0, val_n, 8)])
        nola_correct = int((pred == y_val).sum())
    return dict(train_n=n, max_anchors=k, anchors=len(indices), classes_with_anchors=pseudo.unique().numel(),
                anchor_accuracy=float((audit[indices] == pseudo).float().mean()),
                cde=cde_correct, dino_head=head_correct, nola=nola_correct,
                vs_cde=nola_correct-cde_correct, vs_dino=nola_correct-head_correct, status="ok")

## 4. Run and read the table
`cde`, `dino_head`, and `nola` are correct counts on the **same** `val_n` images. `vs_cde > 0` means full NoLA beats the original CLIP baseline on this subset; `vs_dino > 0` means the final tuning stage adds value over the DINO head. Inspect both plus `anchors`, `classes_with_anchors`, and `anchor_accuracy`: increasing the cap can add coverage, but it can also admit worse pseudo-labels. A result on one seed is exploratory; no universal break-even size follows from this grid.

In [ ]:
from IPython.display import display

results = []
for n, k in itertools.product(train_sizes, anchor_caps):
    print(f"\n--- training images={n}, anchor cap={k} ---", flush=True)
    result = run_setting(n, k)
    results.append(result)
    print(result, flush=True)
results = pd.DataFrame(results)
display(results)